In [11]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, Softplus, PReLU
from torch.utils.data import DataLoader
from torch.optim import Adam

In [12]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu126
CUDA disponível: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [13]:
dataset = load_diabetes()
dataset.data.shape
print(dataset.target.max())
print(dataset.target.min())

346.0
25.0


In [14]:
x = dataset.data
y = dataset.target

X_treino,X_teste, y_treino, y_teste = train_test_split(x,y, test_size=0.2, random_state=42)

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_train_normalizado = scaler.fit_transform(X_treino)
X_test_normalizado = scaler.transform(X_teste)

In [16]:
model = Sequential(
    Linear(10,8),
    ReLU(),
    Linear(8,4),
    ReLU(),
    Linear(4,1)
)
model.to("cuda")
print(model)

Sequential(
  (0): Linear(in_features=10, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)


In [17]:
optmizer = Adam(model.parameters(), lr= 0.001)
criterion = MSELoss().to("cuda")
max_epochs = 300
batch = 8

In [18]:
total_batchs = len(X_treino) // batch
last_batch = len(X_treino) % batch
has_last_b = last_batch > 0

print(total_batchs)
print(last_batch)

44
1


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(
    X_train_normalizado,
    dtype=torch.float32
).to(device)

y_train_tensor = torch.tensor(
    y_treino,
    dtype=torch.float32
).reshape(-1, 1).to(device)

X_test_tensor = torch.tensor(
    X_test_normalizado,
    dtype=torch.float32
).to(device)

In [20]:
def train_model():

    model.train()

    for i_epochs in range(max_epochs):

        for i_batchs in range(total_batchs):

            inicio = i_batchs * batch
            fim = (i_batchs + 1) * batch

            batch_x = X_train_tensor[inicio:fim]
            batch_y = y_train_tensor[inicio:fim]

            optmizer.zero_grad()

            output = model(batch_x)

            loss = criterion(output, batch_y)

            loss.backward()
            optmizer.step()

In [21]:
def eval_model():

    model.eval()

    with torch.no_grad():

        output = model(X_test_tensor)

    output = output.cpu().numpy().ravel()

    print(f"R²: {r2_score(y_teste, output):.4f}")
    print(f"RMSE: {root_mean_squared_error(y_teste, output):.4f}")
    print(f"MSE: {mean_squared_error(y_teste, output):.4f}")

In [22]:
train_model()
eval_model()

R²: 0.4766
RMSE: 52.6578
MSE: 2772.8401
